# Model Experiments
ARIMA regime detection + LSTM forecaster experiments on AAPL.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings("ignore")

df = pd.read_parquet("data/processed/AAPL_features.parquet")
print(f"Dataset: {df.shape[0]} rows, {df.index.min().date()} to {df.index.max().date()}")

## Volatility Regime Detection (HMM)

In [ ]:
from src.models.regime import RegimeDetector

rd = RegimeDetector(n_components=2)
df = rd.fit_predict(df)

regime_counts = df["volatility_regime"].value_counts()
print("Regime distribution:", regime_counts.to_dict())

for regime in [0, 1]:
    sub = df[df["volatility_regime"] == regime]
    print(f"Regime {regime}: mean_vol={sub["realized_vol_10d"].mean()*100:.1f}%  ",
          f"mean_return={sub["log_return"].mean()*100:.4f}%  n={len(sub)}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df["close"],
                         name="AAPL Close", line=dict(color="#89b4fa", width=1)))
colors = {0: "rgba(39,174,96,0.2)", 1: "rgba(231,76,60,0.2)"}
regime = df["volatility_regime"]
prev = regime.iloc[0]; start = df.index[0]
for i in range(1, len(regime)):
    if regime.iloc[i] != prev or i == len(regime)-1:
        fig.add_vrect(x0=start, x1=df.index[i], fillcolor=colors[prev],
                      layer="below", line_width=0)
        start = df.index[i]; prev = regime.iloc[i]
fig.update_layout(template="plotly_dark", title="AAPL Price with Volatility Regimes (Green=Low-Vol, Red=High-Vol)",
                  height=450, showlegend=False)
fig.show()

## Walk-Forward Validation Results

In [ ]:
import os
cv_path = "data/processed/cv_results.csv"
if os.path.exists(cv_path):
    cv = pd.read_csv(cv_path)
    print("Walk-forward results shape:", cv.shape)
    summary = cv.groupby("model")[["rmse","mape","directional_accuracy","mape_improvement_vs_baseline","ic"]].mean().round(4)
    print("
=== Mean metrics across all folds ===")
    print(summary.to_string())
else:
    print("Run: python main.py validate AAPL")

In [ ]:
if os.path.exists(cv_path):
    import plotly.express as px
    ensemble = cv[cv["model"]=="ensemble"].copy()

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["MAPE by Fold", "Directional Accuracy by Fold"])
    for model in ["arima","lstm","ensemble"]:
        sub = cv[cv["model"]==model].sort_values("fold")
        fig.add_trace(go.Scatter(x=sub["fold"], y=sub["mape"], name=model, mode="lines+markers"), row=1, col=1)
        fig.add_trace(go.Scatter(x=sub["fold"], y=sub["directional_accuracy"],
                                 name=model, showlegend=False, mode="lines+markers"), row=1, col=2)
    fig.add_hline(y=50, line_dash="dot", line_color="gray", annotation_text="random (50%)", row=1, col=2)
    fig.update_layout(template="plotly_dark", height=400, title="Walk-Forward Validation — AAPL")
    fig.show()